# Text embeddings: a first look

Three short clinical texts, each turned into **one vector** (an embedding),
and three things done with those vectors: look at them, compare them, and
classify them.

The texts are the opening sentences of three cases of the MultiCaRe sample
used in `../ner/` (cases `PMC5137649_01`, `PMC11722600_01`, `PMC6949661_01`,
`PMC6134951_01`).
Texts 1 and 2 describe a woman with abdominal pain; text 3 describes a man
found confused, possibly after a seizure. Text 4 describes a medication and
its effects.

## 1. Preparation

One model, with its tokenizer. `BAAI/bge-small-en-v1.5` is a small BERT
encoder (33M parameters) trained so that **the vector of its `[CLS]` token
represents the whole text**: texts with similar meanings get vectors that
point in similar directions.

In [12]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
pd.set_option("display.max_columns", 10)          # show the first and last dimensions only

TEXTS = {
    "Text 1": "A 44-year-old woman presented with a 3-day history of right flank and lower quadrant "
              "abdominal pain associated with nausea and constipation.",
    "Text 2": "A 41-year-old female with a history of cesarean section presented to the emergency "
              "department with continuous abdominal pain for 12 h, localized in the right iliac fossa (RIF).",
    "Text 3": "A 23-year-old male was brought to the emergency of Hamad General Hospital in Doha, after "
              "being found on the floor dizzy and confused, with urinary incontinence although no one "
              "witnessed convulsions.",
    "Text 4": "However, 10 days later, she developed a non-pruritic morbilliform rash spreading from trunk "
              "to the entire body, including palms, soles, and face with mild lip and eye swelling."
}

## 2. From text to embedding, one text at a time

The basic cycle, three steps per text:

1. **tokenize** — the text becomes token ids, with `[CLS]` added at the
   start and `[SEP]` at the end;
2. **encode** — the model returns one contextual vector per token;
3. **keep `[CLS]`** — the vector at position 0 is the embedding of the text.

In [13]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).eval()   # eval(): inference only, no training

embeddings = {}

for name, text in TEXTS.items():
    inputs = tokenizer(text, return_tensors="pt")          # 1. text → token ids
    with torch.no_grad():                                  #    (no gradients: we are not training)
        outputs = model(**inputs)                          # 2. token ids → one vector per token
    cls = outputs.last_hidden_state[0, 0]                  # 3. [text 0, position 0] = the [CLS] vector
    embeddings[name] = cls

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    print(f"{name}: {len(tokens)} tokens {tokens[:4]} ... {tokens[-2:]}")
    print(f"        all token vectors: {tuple(outputs.last_hidden_state.shape)}  →  embedding: {tuple(cls.shape)}")

Text 1: 33 tokens ['[CLS]', 'a', '44', '-'] ... ['.', '[SEP]']
        all token vectors: (1, 33, 384)  →  embedding: (384,)
Text 2: 42 tokens ['[CLS]', 'a', '41', '-'] ... ['.', '[SEP]']
        all token vectors: (1, 42, 384)  →  embedding: (384,)
Text 3: 49 tokens ['[CLS]', 'a', '23', '-'] ... ['.', '[SEP]']
        all token vectors: (1, 49, 384)  →  embedding: (384,)
Text 4: 43 tokens ['[CLS]', 'however', ',', '10'] ... ['.', '[SEP]']
        all token vectors: (1, 43, 384)  →  embedding: (384,)


## 3. Looking at the embeddings

The simplest view is a table: **one row per text, one column per
dimension**. Only the first and last dimensions are shown (`...` hides the
rest). A single number means nothing by itself; the meaning is in the
whole row.

In [14]:
table = pd.DataFrame(torch.stack(list(embeddings.values())).numpy(), index=embeddings.keys())
print("shape:", table.shape, "(texts × dimensions)")
table.round(3)

shape: (4, 384) (texts × dimensions)


,0,1,2,3,4,...,379,380,381,382,383
Text 1,-0.131,-0.179,0.051,0.836,-0.018,...,-0.092,-0.141,-0.546,0.248,-0.185
Text 2,0.041,-0.415,-0.067,0.476,0.361,...,0.037,-0.255,-0.296,0.492,-0.530
Text 3,0.157,0.298,-0.094,0.291,0.359,...,-0.172,-0.136,0.266,-0.372,-0.492
Text 4,-0.083,0.315,0.161,0.250,-0.428,...,0.314,-0.602,-0.956,-0.334,0.645


## 4. Comparing texts

The **cosine similarity** of two vectors $a$ and $b$ is
$\dfrac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$: 1 when they point in the same
direction, 0 when they are perpendicular.

In [15]:
names = list(embeddings)

for i, a in enumerate(names):
    for b in names[i + 1:]:
        similarity = F.cosine_similarity(embeddings[a], embeddings[b], dim=0).item()
        print(f"{a} × {b}: {similarity:.3f}")

Text 1 × Text 2: 0.791
Text 1 × Text 3: 0.632
Text 1 × Text 4: 0.575
Text 2 × Text 3: 0.710
Text 2 × Text 4: 0.549
Text 3 × Text 4: 0.521


## 5. Classifying `[CLS]` embeddings with a logistic regression

**Scenario: routing a new case.** A hospital receives case notes and wants
to send each one to the right team. Two notes have already been labeled by
a physician — text 1 as **abdominal** and text 3 as **neurological** — and
text 2 arrives as a new, unlabeled note.

A logistic regression learns one weight per dimension (plus a bias) and
turns the weighted sum of an embedding into a probability. Trained on two
embeddings, it is only a toy, but it is the same construction as BERT's
classification head (`...ForSequenceClassification`): a linear layer on top
of the `[CLS]` vector. Here the encoder stays frozen and only the linear
layer is learned.

In [16]:
X_train = [embeddings["Text 1"].numpy(), embeddings["Text 3"].numpy()]
y_train = ["abdominal", "neurological"]
classifier = LogisticRegression().fit(X_train, y_train)
print("learned:", classifier.coef_.shape[1], "weights + 1 bias\n")

x_new = embeddings["Text 2"].numpy()
print("Text 2 →", classifier.predict([x_new])[0])
for label, p in zip(classifier.classes_, classifier.predict_proba([x_new])[0]):
    print(f"    P({label}) = {p:.3f}")

learned: 384 weights + 1 bias

Text 2 → abdominal
    P(abdominal) = 0.633
    P(neurological) = 0.367


## What to notice

* **Texts of any length, vectors of one size.** The three texts have 33,
  42 and 49 tokens, so the model returns 33, 42 and 49 vectors; keeping
  only `[CLS]` gives each text the same 384 numbers.
* **The numbers are not readable one by one.** No column of the table
  stands for "abdomen" or "seizure"; only comparisons between whole rows
  carry meaning.
* **Texts 1 and 2 are the closest pair** (0.791), and text 1 is farthest
  from text 3 (0.632). Text 2 sits closer to text 3 (0.710) than text 1
  does: both mention *the emergency* and say how the patient arrived,
  while text 1 has only symptoms.
* **Similarities do not go down to 0.** Unrelated clinical sentences still
  share a lot (the genre, "A NN-year-old ..."), so this model's scores live
  in a narrow upper range; what matters is the *order* of the pairs, not the
  value of one.
* **The classifier follows the similarities.** Trained on texts 1 and 3
  alone, it labels text 2 as *abdominal*, with P = 0.633: right, but not
  confident, which is what two training examples can give. More labeled
  notes would sharpen it — the model stays the same, a weighted sum of the
  384 dimensions.
* **Try it:** set `MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"`
  and run everything again. That model was not trained to make `[CLS]`
  summarize a text: all three similarities come out between 0.94 and 0.97,
  and the comparison no longer separates the cases. Its vocabulary knows
  clinical words, but its `[CLS]` vector was never trained for this job.